## clash 정보, plddt 정보 csv파일에 포함하기 

### 필요한 함수 

In [2]:
import numpy as np 
from Bio.PDB import PDBParser

cdr_chothia = {
    "h1": (26, 32),
    "h2": (52, 56),
    "h3": (95, 102),
    "l1": (24, 34),
    "l2": (50, 56),
    "l3": (89, 97)
}

def calculate_b_factor_mean(pdb_file, cdr_type):
    parser = PDBParser(QUIET=True)

    start_residue = cdr_chothia[cdr_type][0]
    end_residue = cdr_chothia[cdr_type][1]

    structure = parser.get_structure("structure", pdb_file)
    
    # 첫 번째 체인 추출
    first_chain = next(iter(structure[0].get_chains()))
    b_factors = []
    for residue in first_chain:
        residue_id = residue.get_id()
        if residue_id[0] == " ":  # 표준 아미노산만 고려
            residue_number = residue_id[1]
            if start_residue <= residue_number <= end_residue:
                # 각 원자의 B-factor 가져오기
                for atom in residue:
                    b_factors.append(atom.get_bfactor())

    return np.mean(b_factors)


### plddt 

In [ ]:
import pandas as pd
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from data.ab_metrics import renumber_pdb  # 또는 renumber_chain

###################################################################################################################
csv_file = '/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2_wt_confidence_lr_1e-3/2025-10-26_10-47-26/epoch=42-step=61619/run_2025-11-08_22-44-09/get_capri/capri_info.csv'
base_dir = Path('/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2_wt_confidence_lr_1e-3/2025-10-26_10-47-26/epoch=42-step=61619/run_2025-11-08_22-44-09')
###################################################################################################################

cdr_list = ['h1', 'h2', 'h3', 'l1', 'l2', 'l3']
df = pd.read_csv(csv_file)

for cdr in cdr_list:
    df[f"{cdr}_plddt"] = None

# ---- 병렬 처리 함수 정의 ----
def process_row(row):
    import traceback
    from data.ab_metrics import renumber_pdb
    try:
        pdb_path = str(base_dir / row['pdb_id'] / row['data_name'] / row['sample_id'] / 'get_capri' / 'sample_1_rechain.pdb')
        output_path = str(base_dir / row['pdb_id'] / row['data_name'] / row['sample_id'] / 'get_capri' / 'sample_1_chothia.pdb')

        # 1️⃣ renumber
        _ = renumber_pdb(pdb_path, out_pdb_file=output_path)

        # 2️⃣ pLDDT 계산
        cdr_plddts = {}
        for cdr in cdr_list:
            cdr_plddts[cdr] = calculate_b_factor_mean(output_path, cdr)

        return (row.name, cdr_plddts, None)

    except Exception as e:
        return (row.name, None, str(e) + "\n" + traceback.format_exc())

# ---- 병렬 실행 ----
results = []
num_workers = min(8, os.cpu_count())  # 병렬 프로세스 수 제한
print(f"🧠 Using {num_workers} parallel workers")

with ProcessPoolExecutor(max_workers=num_workers) as executor:
    futures = [executor.submit(process_row, row) for _, row in df.iterrows()]
    for i, f in enumerate(as_completed(futures), 1):
        idx, cdr_plddts, err = f.result()
        if err:
            print(f"❌ Error at index {idx}: {err}")
            continue
        for cdr in cdr_list:
            df.at[idx, f"{cdr}_plddt"] = cdr_plddts[cdr]
        if i % 10 == 0 or i == len(df):
            print(f"✅ Progress: {i}/{len(df)} done")

# ---- CSV 저장 ----
output_csv = csv_file.replace(".csv", "_with_plddt.csv")
df.to_csv(output_csv, index=False)
print(f"\n✅ Saved updated CSV to: {output_csv}")


## 기존 csv파일에 붙이기 

In [ ]:
import os 
import json 
import pandas as pd 

# 파일 경로
file_a = '/home/kkh517/run-epi-a817668/results/run_20251020/raw_results.csv'
file_b = '/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2_wt_confidence_lr_1e-3/2025-10-26_10-47-26/epoch=42-step=61619/run_2025-11-08_22-44-09/get_capri/capri_info_with_plddt.csv'

# CSV 읽기
df_a = pd.read_csv(file_a)
df_b = pd.read_csv(file_b)

# 열 이름 변경
df_a = df_a.rename(columns={
    'capri_criteria': 'capri_kkh',
    'dockq_score': 'dockq_score_kkh'
})

# 필요한 열만 선택
df_a = df_a[['pdb_id', 'data_name', 'capri_kkh', 'dockq_score_kkh', 'interface_pae']]

# 병합 (inner join: 일치하는 행만)
merged = pd.merge(df_b, df_a, on=['pdb_id', 'data_name'], how='left')

# 결과 저장
output_path = file_b.replace('.csv', '_merged_with_kkh.csv')
merged.to_csv(output_path, index=False)

print(f"✅ Merged CSV saved to: {output_path}")
print(f"✅ Shape: {merged.shape}")

